# TLE fitting with propygator

This notebook walks through **Feature 1.2**: re-expressing a reference orbit — a
high-fidelity numerical propagation, a user-assembled trajectory, or another TLE's
output — as a **shareable TLE**, via Orekit's batch least squares over the SGP4 mean
elements (+ B\*). `fit_tle` returns the fitted `TLE`; its sibling `fit_tle_detailed`
(identical parameters, one shared engine) returns a `FitResult` with the fit
diagnostics.

This is the faithful sibling of `TLE.from_state_unfitted` (notebook 04): that one
stuffs osculating elements into mean-element slots and is deliberately *not*
round-trip-faithful; `fit_tle` actually fits. The defining caveat, worth stating up
front: **the fit is inherently lossy.** SGP4 is a simplified model (J2/J3/J4 zonals
plus a single B\* drag term for the near-Earth branch), so a full-force numerical
orbit can never be reproduced exactly — expect a few hundred meters RMS over a 2-day
LEO span. We will *see* that number below, not just assert it.

We use a **pinned ISS element set** so the notebook runs offline and reproducibly.
As everywhere in propygator, building the `TLE` and the value types is pure-Python;
the JVM starts lazily at the first propagation (`docs/architecture.md` §10).

In [ ]:
import numpy as np

import propygator as pgr

pgr.__version__

## 1. A high-fidelity reference trajectory

The most common use of `fit_tle` is **path (a)** from `docs/architecture.md` §8:
you ran a careful numerical propagation and want to hand the result to tools that
only speak TLE. We build that reference here explicitly — 2 days of `leo_default`
physics (70x70 gravity, Sun/Moon third body, drag, SRP) from an ISS-like state,
with the `high_precision` integrator preset that §1.1 designed for exactly this.

(Passing a `State` to `fit_tle` runs this same internal propagation for you —
`fit_tle(state0, fitting_span=2 * 86400)` — but building the trajectory ourselves
lets us plot the divergence against it afterwards.)

In [ ]:
# A pinned ISS (ZARYA) element set (epoch 2026-06-20) anchors the scenario.
ISS_LINE1 = "1 25544U 98067A   26171.41461525  .00008813  00000+0  16600-3 0  9990"
ISS_LINE2 = "2 25544  51.6327 284.1189 0004557 208.5194 151.5545 15.49333088572250"

tle = pgr.TLE.from_strings(ISS_LINE1, ISS_LINE2, name="ISS (ZARYA)")

# The reference initial state: the TLE's own state at epoch, in EME2000.
state0 = pgr.propagate_tle(tle, 60, output_step=60)[0].to_frame(pgr.Frame.EME2000)

reference = pgr.propagate_numerical(
    state0,
    duration=2 * 86400,
    output_step=600,
    force_models=pgr.ForceModelConfig.leo_default(),
    spacecraft=pgr.SpacecraftConfig(),
    integrator=pgr.IntegratorConfig.high_precision(),
    progress=False,
)
print(len(reference), "samples over", reference.frame)

## 2. Fit the TLE

`fit_tle(reference, *, fitting_span=2 * 86400, ..., fit_bstar=True, norad_id=None,
name=None, progress=True)` seeds a template TLE at the reference start (a
fixed-point osculating-to-mean inversion of the first sample), then runs a
Levenberg-Marquardt batch least squares over ~300 evenly subsampled position/velocity
measurements. The fitted epoch is the **reference start**, and with
`fit_bstar=True` (the default — right for LEO, where a 2-day span makes drag
observable) B\* is estimated alongside the six mean elements.

Identity fields (catalog number, name, designator...) are bookkeeping no trajectory
can supply: they resolve as *explicit kwarg -> inherited from `initial_guess` ->
placeholder*. Physics is fitted-or-zeroed — the mean-motion derivatives are always
`0.0` (SGP4 ignores them).

By default the fit streams `iter N | rms ...` progress lines to stderr; we pass
`progress=False` to keep the notebook tidy.

In [ ]:
fitted = pgr.fit_tle(reference, norad_id=25544, name="ISS (ZARYA)", progress=False)

print("fitted:", fitted.line1, fitted.line2, sep="\n        ")
print("source:", tle.line1, tle.line2, sep="\n        ")

The fitted line 2 sits close to the source element set (same orbit, after all) but
not on it: the fitted elements describe **this 2-day full-force arc**, not the
catalog's radar fit. Note in particular B\* on line 1 — the fitted value is a *fit
residual* that absorbs the drag of this arc under SGP4's crude drag model, not a
physical ballistic coefficient (and not the catalog's `16600-3`).

## 3. The lossiness, made visible

The fitted TLE is a *shareable approximation*. Propagate it back over the fitted
span and plot the position error against the numerical reference — this is the
honest cost of the SGP4 re-expression.

In [ ]:
back = pgr.propagate_tle(
    fitted, 2 * 86400, output_step=600, start=reference.start_epoch
).to_frame(pgr.Frame.EME2000)

divergence_m = np.linalg.norm(back.positions - reference.positions, axis=1)
rms_m = float(np.sqrt(np.mean(divergence_m**2)))
print(f"propagate-back residual: {rms_m:.0f} m RMS, {divergence_m.max():.0f} m max")

In [ ]:
import matplotlib.pyplot as plt

hours = np.array(
    [s.epoch.seconds_since(reference.start_epoch) / 3600.0 for s in reference]
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hours, divergence_m, linewidth=1.5)
ax.set_xlabel("hours since fit epoch")
ax.set_ylabel("position error (m)")
ax.set_title("Fitted TLE vs the numerical reference (the lossiness)")
ax.grid(alpha=0.3)
fig.tight_layout()

A few hundred meters RMS over two days (the validation suite pins ~495 m RMS /
~1.1 km max for this scenario) — excellent by TLE standards, where against *reality*
errors are typically kilometers, but never zero: the oscillation is the full-force
physics SGP4 cannot represent. A longer `fitting_span` averages over more of the
unmodeled dynamics (lower peak error growth beyond the span, higher in-span
residual); a shorter one fits tighter but degrades faster outside it.

## 4. The fit diagnostics: `fit_tle_detailed`

`fit_tle` is a thin wrapper over `fit_tle_detailed`, which returns a frozen
`FitResult` carrying the fitted TLE plus the diagnostics: iterations, evaluations,
the final measurement RMS, and the **per-measurement position residuals** with
their epochs. The fit is deterministic, so re-running it reproduces the same TLE
exactly. (There is deliberately no `converged` flag — a non-converged fit raises
`TLEFitError` with no partial result, so a `FitResult` only exists for converged
fits.)

Note the distinction: `result.rms_m` is the observed-vs-estimated RMS over the
~300 *fit measurements* at convergence — the same number the final progress line
quotes — while the propagate-back curve above is the error over the full output
grid. For this noise-free reference they land close together.

In [ ]:
result = pgr.fit_tle_detailed(
    reference, norad_id=25544, name="ISS (ZARYA)", progress=False
)

assert result.tle == fitted  # one engine: fit_tle(...) is fit_tle_detailed(...).tle
print(
    f"converged in {result.iterations} iterations "
    f"({result.evaluations} evaluations), rms {result.rms_m:.0f} m "
    f"over {len(result.residuals_m)} measurements"
)

In [ ]:
meas_hours = np.array(
    [
        e.seconds_since(result.measurement_epochs[0]) / 3600.0
        for e in result.measurement_epochs
    ]
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(meas_hours, result.residuals_m, linewidth=1.5)
ax.set_xlabel("hours since fit epoch")
ax.set_ylabel("measurement residual (m)")
ax.set_title("Per-measurement residuals at convergence (FitResult)")
ax.grid(alpha=0.3)
fig.tight_layout()

## 5. Residual *structure*: the RIC read

Norms hide the most diagnostic information in the residual set — *structure*.
The 2026-07-19 §1.2 amendment exposes it: `residuals_ric_m` holds the **signed**
position residuals decomposed onto radial / along-track / cross-track axes built
from the observed PV at each measurement (sign convention: **observed −
estimated**, so a positive along-track entry means the reference runs ahead of
the fitted TLE), and `velocity_residuals_ms` the per-measurement velocity
residual norms. The projection is orthonormal, so RIC row norms reproduce
`residuals_m` exactly.

The read is a two-way fork:

- **Periodic** (once-per-rev) structure — SGP4's short-period representation
  error. Irreducible: the fit is as good as SGP4 gets.
- **Secular** (a ramp, along-track especially) — a dynamics mismatch: drag/B\*
  wrong for the arc. Revisit `fit_bstar` or the `fitting_span`.

Below, the fit against our full-force reference shows the periodic signature,
dominated by along-track — the anatomy of section 3's lossiness. (The raw TEME
residual vectors are deliberately *not* stored: they mix orbit-frequency
rotation into every component, so no read exists there. If you need them,
recover exactly as in section 3 — propagate the fitted TLE at the measurement
epochs and subtract your reference.)

In [ ]:
ric = result.residuals_ric_m  # columns: [radial, along-track, cross-track]

fig, ax = plt.subplots(figsize=(8, 4))
for i, label in enumerate(["radial", "along-track", "cross-track"]):
    ax.plot(meas_hours, ric[:, i], linewidth=1.0, label=label)
ax.set_xlabel("hours since fit epoch")
ax.set_ylabel("signed position residual (m)")
ax.set_title("Residual structure in RIC axes (observed - estimated)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

rms_ric = np.sqrt(np.mean(ric**2, axis=0))
print(
    f"RIC RMS: radial {rms_ric[0]:.0f} m | along-track {rms_ric[1]:.0f} m "
    f"| cross-track {rms_ric[2]:.0f} m"
)
vel = result.velocity_residuals_ms
print(f"velocity residual norms: {vel.min() * 1e3:.2f}-{vel.max() * 1e3:.2f} mm/s")

# The consistency invariant: RIC row norms are residuals_m.
assert np.allclose(np.linalg.norm(ric, axis=1), result.residuals_m)

## 6. Is B\* actually constrained? The covariance read

`FitResult` also carries the estimator's **raw** parameter covariance (the
2026-07-18 §1.2 amendment): `covariance` over `parameter_names` — the
**Cartesian TEME state at the fitted epoch** (`Px Py Pz` in m, `Vx Vy Vz` in
m/s) plus `BSTAR` when `fit_bstar=True`, the only basis Orekit's TLE builder
estimates in. No mean-element drivers exist, which is why B\* is the *only*
parameter a fitting strategy can hold or transplant — a fact section 9 builds
on. Alongside it: the derived `sigmas` property (its diagonal, 1-sigma) and
`sigma0`, the a-posteriori variance factor.

Read them honestly: the sigmas are **conditioning indicators**, not absolute
uncertainty — computed under the fit's internal 1 m / 1 mm/s weighting against
a *systematic* SGP4 residual; multiply by `sigma0` for the residual-scaled
form. And **no single-fit number says "B\* is unconstrained"**: on a weak-drag
arc the fitted B\* inflates in step with its own sigma. The honest reads are
*comparative* — across configurations, or against a physically plausible B\* —
and section 9 turns exactly that into a decision rule.

(Power-user notes: any orbital-element sigma is one delta-method projection of
`covariance` away, and the correlation matrix is a one-liner — but corr(B\*, n)
≈ −0.97 on weak *and* strong arcs is **structural** to TLE fitting, both
parameters acting along-track, which is why no correlation property shipped.
See the §1.2 amendment note.)

In [ ]:
names = result.parameter_names
sig = result.sigmas
print(f"sigma0 = {result.sigma0:.3g}  (residuals vs the assumed 1 m / 1 mm/s sigmas)")
for name, raw, scaled in zip(names, sig, sig * result.sigma0):
    print(f"  sigma({name:5s})  raw {raw:10.3e}   x sigma0 {scaled:10.3e}")

# The comparative B* read: a 2-day LEO arc makes drag observable, so the scaled
# sigma(B*) sits well below a plausible B* scale — the ISS catalog's 1.66e-4.
k = names.index("BSTAR")
print(f"\nscaled sigma(B*) / catalog B* = {sig[k] * result.sigma0 / 1.66e-4:.3f}")

## 7. The known-exact case: refitting SGP4's own output

When the reference *is* an SGP4 trajectory (**path (c)**), the fit has an exact
answer — and recovers it. This separates the estimation plumbing from the model
lossiness: the least squares is exact; SGP4's expressiveness is the only limit.
Passing the source TLE as `initial_guess` also donates its identity fields
(designator, element-set number, ...) to the fitted line.

In [ ]:
ref_sgp4 = pgr.propagate_tle(tle, 2 * 86400, output_step=60)
refit = pgr.fit_tle(ref_sgp4, initial_guess=tle, progress=False)

print("refit:  ", refit.line2)
print("source: ", tle.line2)

back_sgp4 = pgr.propagate_tle(
    refit, 2 * 86400, output_step=60, start=ref_sgp4.start_epoch
)
self_rms = float(
    np.sqrt(
        np.mean(np.linalg.norm(back_sgp4.positions - ref_sgp4.positions, axis=1) ** 2)
    )
)
print(f"self-fit propagate-back residual: {self_rms:.2g} m RMS")

## 8. Failure is honest

A fit that cannot converge within `max_iterations` raises `TLEFitError` — carrying
the iteration count and the last RMS, never a raw Java trace, and **no partial
TLE**. (Bad *inputs* — a non-positive `fitting_span`, too few samples, an unbound
orbit — raise `ValueError` pre-flight instead, before the JVM even starts.)

In [ ]:
try:
    pgr.fit_tle(ref_sgp4, max_iterations=1, progress=False)
except pgr.TLEFitError as exc:
    print("TLEFitError:", exc)

## 9. From diagnostics to strategy: the fitting playbook

Everything above reads *one* fit. The diagnostics earn their keep when they
help you **choose a strategy** across fits:
`docs/tle-fitting-playbook-updated.md` distills a single recipe,
validated against GRACE-FO truth across ten
quiet-to-storm windows (`experiments/extended-validation/`). The trap it
guards against: **in-arc RMS is an anti-signal** — the worst-forecasting
configurations often post the *best* fit RMS, because a free B\* on a short
arc absorbs along-track error into a garbage drag coefficient.

**The recipe, no branching:**

1. Fit B\* free on the freshest 2-day arc.
2. Transplant that B\* onto a fresh 1-day refit (`initial_guess=`,
   `fit_bstar=False`).

This one configuration tied or beat every alternative tested (a `B*=0`
refit, a fresh free-B\* refit) in **30 of 30** window-by-day cells across
four solar-activity bands — including quiet, where the original
single-satellite evidence had recommended `B*=0` instead.

*The original recipe gated between three strategies using two diagnostics
from `FitResult` (r and s, from staged 2-day/3-day fits). Tested at scale
the gate added no measurable value over always transplanting — one arm did
the job everywhere, so r and s are retired from the current recipe. The
original gate is archived unchanged in `experiments/tle-fit-strategy/`.*

In [ ]:
# The recipe: 2-day free-B* fit -> transplant its B* onto a fresh 1-day refit.
# fitting_span clips the LEADING portion of a trajectory, so to share an END
# (the freshest data, as an operational refit would) we slice both arcs from
# a common 3-day reference.
reference_3d = pgr.propagate_numerical(
    state0,
    duration=3 * 86400,
    output_step=600,
    force_models=pgr.ForceModelConfig.leo_default(),
    spacecraft=pgr.SpacecraftConfig(),
    integrator=pgr.IntegratorConfig.high_precision(),
    progress=False,
)


def tail(traj: pgr.Trajectory, span_s: float) -> pgr.Trajectory:
    epochs = [st.epoch for st in traj]
    n = sum(1 for e in epochs if traj.end_epoch.seconds_since(e) <= span_s)
    return pgr.Trajectory.from_arrays(
        epochs[-n:], traj.positions[-n:], traj.velocities[-n:], traj.frame
    )


def bstar_of(t: pgr.TLE) -> float:
    """B* from line-1 cols 54-61 (sign + 5-digit mantissa + signed exponent)."""
    f = t.line1[53:61]
    return (-1.0 if f[0] == "-" else 1.0) * (int(f[1:6]) / 1e5) * 10.0 ** int(f[6:8])


# Step 1: fit B* free on the freshest 2-day arc.
staging = pgr.fit_tle_detailed(
    tail(reference_3d, 2 * 86400),
    fitting_span=2 * 86400,
    norad_id=25544,
    progress=False,
)

# Step 2: transplant that B* onto a fresh 1-day refit.
recipe = pgr.fit_tle_detailed(
    tail(reference_3d, 86400),
    fitting_span=86400,
    initial_guess=staging.tle,  # donates B*(2d)...
    fit_bstar=False,  # ...and this holds it
    norad_id=25544,
    progress=False,
)

print(f"B*(2d staging) = {bstar_of(staging.tle):.3e}")
print(f"B*(carried)    = {bstar_of(recipe.tle):.3e}  (= B*(2d) exactly)")
print(
    f"1-day refit epoch: reference start + "
    f"{recipe.tle.epoch.seconds_since(reference_3d.start_epoch) / 3600:.0f} h "
    f"(the freshest day's start) | rms {recipe.rms_m:.0f} m"
)

A few rules of thumb from the measured evidence
(`docs/tle-fitting-playbook-updated.md`), unchanged by the extended study:

- **Never trust a 1-day free-B\* fit's B\*** — garbage in every regime
  measured. Always hold something: a transplant, or `B*=0` as a last resort
  with no staging arc available.
- **The physical formula B\* = ½·ρ₀·Cd·A/m is never the answer** — it
  injects fake decay (up to 76 km at +4 days, measured in the original
  single-anchor test).
- **Epoch placement is irrelevant** — a TLE is a trajectory;
  reparameterizing the same fit at the arc end changes forecasts by < 2 m.
  Anchor your *data*, not the epoch.

One demo-honesty note: the evidence above is regime-validated (ten windows,
four solar-activity bands) but still **one body** (GRACE-FO) — not yet
tested on a second satellite.

---

That's the Feature 1.2 loop: **reference → `fit_tle` / `fit_tle_detailed` →
`FitResult` → diagnostics (residual structure, covariance) → strategy** (section 9's
recipe: free-B\* on a 2-day arc, transplanted onto a fresh 1-day refit).

The SGP4/SDP4 branch follows automatically from the fitted mean motion, exactly as
in `propagate_tle` — a Molniya self-fit recovers its source in the validation suite.
LEO is the *validated* domain; elsewhere, `TLEFitError` is an honest outcome, not a
silent wrong answer. The full contract — field policy, failure modes, the covariance
and residual-diagnostics amendments — is in `docs/features.md` §1.2.